In [ ]:
import optuna
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import lightgbm as lgb

from datetime import datetime
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error

/Users/saraziadat/miniconda3/envs/bbb_exc_score_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
SEED = 15

# Set dataset, data set (model) options: full_dataset, maccs, mordred, ecfp, ecfp_mordred, ecfp_maccs, mordred_maccs
model = 'baseline'
full_dataset_path = '../datasets/b3db_drug_like_labelled.csv'

# Set output file paths
output_optuna_db = f'../model_results/{model}/lgb_b3db_drug_like_labelled_{SEED}_{model}_final' 
optuna_study_name_root = f'lgb_b3db_drug_like_labelled_{SEED}_{model}_final' # mordred_maccs_
training_output_file_path = f'../model_results/{model}/training_output_values_{SEED}_{model}_final.csv'
shap_output_file_path = f'../model_results/{model}/training_shap_output_values_{SEED}_{model}_final.csv'
model_root_path = f'../model_results/{model}/trained_lgb_b3db_drug_like_labelled_{SEED}_{model}_final'

In [ ]:
# Import dataset
full_df = pd.read_csv(full_dataset_path)

smiles = full_df.loc[:, 'SMILES']
bbb_class = full_df.loc[:, 'Class']
logbb = full_df.loc[:, 'logBB']

starting_col = list(full_df.columns).index('Class') + 1
X_full = full_df.iloc[:, starting_col:]

bbb_class = np.array(bbb_class)
logbb = np.array(logbb)

X_full = X_full
col_names = list(full_df.columns)[starting_col:]

X_full = X_full.values
X_to_drop = pd.DataFrame(data=X_full, columns=col_names)
X_full_no_nan = X_to_drop.dropna(axis=1)
X_full_no_nan = X_full_no_nan.loc[:, X_full_no_nan.std() != 0]

X_all = X_full_no_nan.values

In [ ]:
# Training and cross validation: 

# could also change seeds individually. 

n_outer = 5

outer_cv = StratifiedKFold(n_splits=n_outer, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# TODO: Write more efficiently. 

full_set_scores = [0] * n_outer
best_params_all = []
balanced_acc_outer, prec_outer, recall_outer, rocauc_outer, prauc_outer, kappa_outer, mcc_outer, f1_outer  = [], [], [], [], [], [], [], []
mae_outer, mse_outer, mde_outer, r2_outer, pearson_outer = [],[],[],[],[]
all_vals = []
shap_values_all = []
pearson_sim_all = []
all_max_sim = []
abs_error_all = []
all_preds = []
all_trues_r = []
all_trues_c = []

shap_means_all_outers=[]

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
print('Timestamp included in optuna db name: ', timestamp)

count_outer = 0
for train_idx, test_idx in outer_cv.split(X_all, bbb_class):
    count_outer +=1
    X_train, X_test = X_all[train_idx], X_all[test_idx]
    y_train_c, y_test_c = bbb_class[train_idx], bbb_class[test_idx]
    # y_train_r, y_test_r = np.array([np.nan]*len(bbb_class))[train_idx], np.array([np.nan]*len(bbb_class))[test_idx], 
    y_train_r, y_test_r = logbb[train_idx], logbb[test_idx]
    smiles_train, smiles_test = smiles[train_idx], smiles[test_idx]

    def objective(trial):
        
        mse = []
        count_inner = 1
        for inner_train_idx, val_idx in inner_cv.split(X_train, y_train_c):
            X_inner_train, X_val = X_train[inner_train_idx], X_train[val_idx]
            y_inner_train_c, y_val_c = y_train_c[inner_train_idx], y_train_c[val_idx]
            y_inner_train_r, y_val_r = y_train_r[inner_train_idx], y_train_r[val_idx]

            train_data = lgb.Dataset(X_inner_train, label=y_inner_train_r)
            val_data = lgb.Dataset(X_val, label=y_val_r)

            params = {
            "objective": "mse",
            "metric": "None",           
            "boosting_type": "gbdt",
            "verbosity": -1,
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
            "num_leaves": trial.suggest_int("num_leaves", 20, 150),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            'seed': (SEED + count_outer)* count_inner,
            'feature_fraction_seed': (SEED + count_outer)* count_inner,
            'bagging_seed': (SEED + count_outer)* count_inner,
            'drop_seed': (SEED + count_outer)* count_inner,
            'data_random_seed': (SEED + count_outer)* count_inner,
            'extra_seed': (SEED + count_outer)* count_inner,
            'feature_fraction': 1.0,  
            'bagging_fraction': 1.0,  
            'bagging_freq': 0,        
            'force_col_wise': True,  
            'num_threads': 1,  
        }
            model = lgb.train(
                params,
                train_data,
                num_boost_round=1000,
                valid_sets=[val_data],
            )

            y_val_pred = model.predict(X_val, num_iteration=model.best_iteration)
            mse.append(mean_squared_error(y_val_r[~np.isnan(y_val_r)], y_val_pred[~np.isnan(y_val_r)]))
            count_inner += 1

        return np.mean(mse)

    # Run Optuna for inner loop
    study = optuna.create_study(direction="minimize", 
                                storage=f'sqlite:///{output_optuna_db}_{timestamp}.db', 
                                study_name=f'{optuna_study_name_root}_outerfold_{count_outer}', 
                                sampler=optuna.samplers.TPESampler(seed=SEED*count_outer), 
                                pruner=optuna.pruners.BasePruner, 
                                )
    
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    # Train final model on full inner train set using best params
    best_params = study.best_trial.params
    best_params.update({
        "objective": "mse",
        "metric": "None",
        "verbosity": -1,
        "boosting_type": "gbdt"
    })

    best_params_all.append(best_params)

    train_data = lgb.Dataset(X_train, label=y_train_r)
    test_data = lgb.Dataset(X_test, label=y_test_r)

    final_model = lgb.train(
        best_params,
        train_data,
        num_boost_round=1000,
        valid_sets=[test_data],
    )

    
    y_test_pred = final_model.predict(X_test, num_iteration=final_model.best_iteration)
    all_preds.extend(y_test_pred)
    all_trues_r.extend(y_test_r)
    all_trues_c.extend(y_test_c)


    # Determine predictions that include the training dataset for that outerfold: 
    full_set_scores[count_outer-1] = final_model.predict(X_all, num_iteration=final_model.best_iteration)

    # Get SHAP values
    explainer = shap.Explainer(final_model)
    shap_vals = explainer(X_test)

    # Calculate mean SHAP values for each fold
    shap_means_all_outers.append(np.mean(shap_vals.values, axis=0))

    # Save the best model from each outerfold: 
    final_model.save_model(f'{model_root_path}_{count_outer}.txt')


shap_df = pd.DataFrame(shap_means_all_outers, columns=list(X_full_no_nan.columns))
shap_df.to_csv(f"{shap_output_file_path}")

features_to_output = [logbb, bbb_class, all_preds, all_trues_r, all_trues_c] +  [full_set_scores[i] for i in range(n_outer)]
col_names_to_output = ['logbb','bbb_class', 'all_preds', 'all_trues_r', 'all_trues_c'] + [f'full_set_scores_fold_{i}' for i in range(n_outer)]


output_vals = pd.DataFrame(np.transpose(features_to_output), columns= col_names_to_output)
output_vals.to_csv(training_output_file_path)


Timestamp included in optuna db name:  20251115_113047


[I 2025-11-15 11:30:48,189] A new study created in RDB with name: lgb_b3db_drug_like_labelled_15_baseline_final_outerfold_1
Best trial: 0. Best value: 0.350578:   3%|▎         | 1/30 [00:27<13:19, 27.56s/it]

[I 2025-11-15 11:31:15,751] Trial 0 finished with value: 0.3505775641437191 and parameters: {'learning_rate': 0.2561571322078878, 'num_leaves': 43, 'max_depth': 3, 'min_child_samples': 21, 'subsample': 0.637700464303206, 'colsample_bytree': 0.7650001124477126}. Best is trial 0 with value: 0.3505775641437191.


Best trial: 1. Best value: 0.34985:   7%|▋         | 2/30 [01:04<15:31, 33.27s/it] 

[I 2025-11-15 11:31:53,029] Trial 1 finished with value: 0.34984958342786004 and parameters: {'learning_rate': 0.09871648556341564, 'num_leaves': 59, 'max_depth': 4, 'min_child_samples': 16, 'subsample': 0.9588149489182812, 'colsample_bytree': 0.6320734266266868}. Best is trial 1 with value: 0.34984958342786004.


...

[I 2025-11-15 14:54:06,596] Trial 27 finished with value: 0.3551719738924958 and parameters: {'learning_rate': 0.08995619939985282, 'num_leaves': 97, 'max_depth': 11, 'min_child_samples': 33, 'subsample': 0.5761015116488175, 'colsample_bytree': 0.6944167651743305}. Best is trial 24 with value: 0.3501488157921435.


Best trial: 24. Best value: 0.350149:  97%|█████████▋| 29/30 [33:24<01:11, 71.35s/it]

[I 2025-11-15 14:55:06,306] Trial 28 finished with value: 0.3543118536839997 and parameters: {'learning_rate': 0.10166921605493989, 'num_leaves': 75, 'max_depth': 8, 'min_child_samples': 41, 'subsample': 0.6603529075729937, 'colsample_bytree': 0.707347364149122}. Best is trial 24 with value: 0.3501488157921435.


Best trial: 24. Best value: 0.350149: 100%|██████████| 30/30 [34:34<00:00, 69.15s/it]


[I 2025-11-15 14:56:16,227] Trial 29 finished with value: 0.3738564586095961 and parameters: {'learning_rate': 0.18128382942635185, 'num_leaves': 110, 'max_depth': 11, 'min_child_samples': 46, 'subsample': 0.5630457166772733, 'colsample_bytree': 0.7948015378774205}. Best is trial 24 with value: 0.3501488157921435.
